In [15]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor 
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder

Data Importation

In [16]:
X_train = pd.read_csv("data/challenge_train_features.csv", index_col=0)
y_train = pd.read_csv("data/challenge_train_revenue.csv", index_col=0)
X_test = pd.read_csv("data/challenge_test_features.csv", index_col=0)

Transformations of data using pandas (for date) and for y_train (log transformation)

In [17]:
X_train["date_format"] = pd.to_datetime(X_train["date"], format="%m/%d/%y")
X_train.loc[X_train["date_format"].dt.year > 2025, "date_format"] -= pd.offsets.DateOffset(years=100)
X_train["year"] = X_train["date_format"].dt.year
X_train["month"] = X_train["date_format"].dt.month

X_test["date_format"] = pd.to_datetime(X_test["date"], format="%m/%d/%y")
X_test.loc[X_test["date_format"].dt.year > 2025, "date_format"] -= pd.offsets.DateOffset(years=100)
X_test["year"] = X_test["date_format"].dt.year
X_test["month"] = X_test["date_format"].dt.month

y_train_log = np.log1p(y_train.values)

Transformations of data using sklearn pipeline

In [ ]:
#Transformation functions


def clip_popularity(X):
    X = X.copy()
    X['popularity_score'] = X['popularity_score'].clip(upper=20)
    return X[['popularity_score']]

def log_budget(X):
    X = X.copy()
    X['budget'] = np.log1p(X['budget'].clip(lower=0))
    return X[['budget']]

def collection_to_binary(X):
    return X.notna().astype(int).to_numpy().reshape(-1,1)

def english_to_binary(X):
    return (X == 'en').astype(int).to_numpy().reshape(-1,1)

def US_to_binary(X):
    return (X == 'US').astype(int).to_numpy().reshape(-1,1)

#We create the preprocessing pipeline
#Columns to be transformed
num_cols = ['budget', 'popularity_score']
cat_cols = ['collection', 'language', 'country','month']

preprocessor = ColumnTransformer(
    transformers=[
        ('budget', Pipeline([
            ('log', FunctionTransformer(log_budget, validate=False)),
            ('scaler', StandardScaler())
        ]), ['budget']),
        
        ('popularity', Pipeline([
            ('clip', FunctionTransformer(clip_popularity, validate=False)),
            ('scaler', StandardScaler())
        ]), ['popularity_score']),
        
        ('collection_bin', FunctionTransformer(collection_to_binary, validate=False), ['collection']),
        ('language_bin', FunctionTransformer(english_to_binary, validate=False), ['language']),
        ('country_bin', FunctionTransformer(US_to_binary, validate=False), ['country']),
        ('month_cat', OneHotEncoder(handle_unknown='ignore'), ['month'])
    ],
    remainder='drop'
)


pipeline = Pipeline([
    ('preprocessor', preprocessor)
])

#We apply the pipeline to the training dataset


X_train_transformed = pipeline.fit_transform(X_train)

#We apply the pipeline to the test dataset
X_test_transformed = pipeline.transform(X_test)


Model

In [24]:
model = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    min_child_weight=1.0,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_transformed, y_train_log)

y_pred_log= model.predict(X_test_transformed)

y_test_pred = np.expm1(y_pred_log).clip(0, None)


Saving in a text file

In [25]:
pred_str = ",".join([str(int(p)) for p in y_test_pred])  

with open("test4.txt", "w") as f:
    f.write(pred_str)